#Mount Google Drive to access your data



In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


The following code uses a python library named 'torchsample'. This is not installed in Google Colab. We can import it by running the commands in the following cell. The exclamation mark communicates to Google Colab to run the commands in the terminal rather than in Python in the current notebook.


In [ ]:
!pip install -e git+https://github.com/ncullen93/torchsample.git#egg=torchsample
!pip install visdom
!pip install nibabel
!pip install h5py
!pip install tensorboardX

Obtaining torchsample from git+https://github.com/ncullen93/torchsample.git#egg=torchsample
  Cloning https://github.com/ncullen93/torchsample.git to ./src/torchsample
  Running command git clone --filter=blob:none --quiet https://github.com/ncullen93/torchsample.git /content/src/torchsample
  Resolved https://github.com/ncullen93/torchsample.git to commit ecd3547c79643687412cb24aa872b4923a4fb865
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
Discarding git+https://github.com/ncullen93/torchsample.git#egg=torchsample: Requested nitrain from git+https://github.com/ncullen93/torchsample.git#egg=torchsample has inconsistent name: expected 'torchsample', but metadata has 'nitrain'
ERROR: Could not find a version that satisfies the requirement torchsample (unavailable) (from versions: 0.1.0)
ERROR: No matching distribution found for

In [ ]:
#import all libraries
import torch.optim as optim
import torch
import torch.nn as nn
from torchvision import models
import numpy as np
import os
import sys
import pickle
import torch.nn.functional as F
import torch.utils.data as data
import pandas as pd
from torch.autograd import Variable
from torchvision import transforms
from tensorboardX import SummaryWriter
import math
from sklearn import metrics

#Define your model
The model is defined in the class 'Net'. The 'init' function initialises the architecture of the model.

The line of code; ```self.pretrained_model = models.resnet18(pretrained=True)``` initialises a pre-trained ResNet18, pre-trained on the ImageNet Dataset. This initialises the weights of the model with the weights for a ResNet18 model that was trained on the ImageNet dataset. This speeds up training.

The line of code ```self.classifer = nn.Linear(1000, 1)``` is a fully connected layer that makes the final prediction.

After the model is initialised, the forward function is called iteratively throughout the training process. The output size of each line is shown in the code.

More information con defining models can be found at https://pytorch.org/vision/stable/models.html

In [ ]:
models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
list( models.resnet18(pretrained=True).children())[:-1]

[Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False),
 BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
 ReLU(inplace=True),
 MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False),
 Sequential(
   (0): BasicBlock(
     (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
   )
   (1): BasicBlock(
     (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), pad

In [ ]:
# Upgraded MRNet Architecture supporting multiple backbones and pooling strategies
class Net(nn.Module):
    def __init__(self, backbone='resnet18', pooling='attention', dropout_prob=0.5):
        super().__init__()
        self.backbone_name = backbone
        self.pooling = pooling

        # Load pre-trained backbone
        if backbone == 'resnet18':
            resnet = models.resnet18(pretrained=True)
            self.pretrained_model = nn.Sequential(*list(resnet.children())[:-1])
            self.feature_dim = 512
        elif backbone == 'resnet34':
            resnet = models.resnet34(pretrained=True)
            self.pretrained_model = nn.Sequential(*list(resnet.children())[:-1])
            self.feature_dim = 512
        elif backbone == 'resnet50':
            resnet = models.resnet50(pretrained=True)
            self.pretrained_model = nn.Sequential(*list(resnet.children())[:-1])
            self.feature_dim = 2048
        elif backbone == 'efficientnet_b0':
            try:
                effnet = models.efficientnet_b0(pretrained=True)
                # EfficientNet has a features block, then avgpool, then classifier. We only want features.
                self.pretrained_model = effnet.features
                self.feature_dim = 1280
            except AttributeError:
                # Fallback to resnet18 if efficientnet is not available in older torchvision versions
                resnet = models.resnet18(pretrained=True)
                self.pretrained_model = nn.Sequential(*list(resnet.children())[:-1])
                self.feature_dim = 512
        else:
            raise ValueError(f"Unknown backbone: {backbone}")

        # Attention mechanism to weigh slices dynamically
        if pooling == 'attention':
            self.attention = nn.Sequential(
                nn.Linear(self.feature_dim, 128),
                nn.Tanh(),
                nn.Linear(128, 1)
            )

        # Solve Problem 3: Dropout before classifier head to prevent overfitting
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_prob),
            nn.Linear(self.feature_dim, 1)
        )

    def forward(self, x):
        # input size of x (1, s, 3, 256, 256) where s is the number of slices in one MRI
        x = torch.squeeze(x, dim=0) # output size (s, 3, 256, 256)
        x = self.pretrained_model(x) # output size (s, feature_dim, H, W) or (s, feature_dim, 1, 1)

        # If output from backbone is convolutional feature maps (e.g. from efficientnet), pool spatially first
        if len(x.shape) == 4:
            x = nn.AdaptiveAvgPool2d((1, 1))(x)

        x = x.view(x.size(0), -1) # output size (s, feature_dim)

        if self.pooling == 'attention':
            # Calculate attention weights for each slice
            a = self.attention(x) # output size (s, 1)
            a = torch.softmax(a, dim=0) # output size (s, 1)
            # Weighted sum of slices
            output = torch.sum(x * a, dim=0, keepdim=True) # output size (1, feature_dim)
        elif self.pooling == 'max':
            output = torch.max(x, dim=0, keepdim=True)[0] # output size (1, feature_dim)
        elif self.pooling == 'avg':
            output = torch.mean(x, dim=0, keepdim=True) # output size (1, feature_dim)
        else:
            raise ValueError(f"Unknown pooling method: {self.pooling}")

        output = self.classifier(output) # output size (1, 1)

        return output.squeeze(1)


In [ ]:
# Net class is defined and customized in Cell 10. This cell is cleared to avoid duplicate definition issues.

In [ ]:
mod=Net()
mod

Net(
  (pretrained_model): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=

TO NOTE:
Models defined in Pytorch expect 2D image data in the dimensions (batch size, channels (colours), height of the image, width of the image)

#Create Dataloader
The 'init' function initialises the dataloader. This class is responsible for loading the datasets. It takes the 'root_dir', 'task', 'plane', 'train' and 'transform' as input parameters.
root_dir - the directory to where the data is stored.

task - whether the model is being trained to detect acl tears, meniscus tears or abnormalities. Possible values are 'acl', 'meniscus' or 'abnormal'.

plane - whether the model is being trained on axial, coronal or sagittal data. Possible values are 'axial', 'coronal' or 'sagittal'.

train - is this the dataloader for the training data or the validation data. Possible values are 'True' to load training data or 'False' to load validation data.

transform - a compose function for performing transformations to the images.

The init function creates 1) a list of paths to each MRI, 2) a corresponding list of labels that are either ones or zeros and 3) weights.


---



The __len__ function returns the length of the dataset.


---
The __getitem__ function is iteratively called throughout the training process. It takes an index as a input parameter. It loads the MRI at the given index from the list of paths defined in the init function. It also returns the label and weight for the MRI at that index.



In [ ]:
from PIL import Image

class Dataset(data.Dataset):
    def __init__(self, root_dir, task, plane, train=False, transform=None):
        super().__init__()
        self.task = task
        self.plane = plane
        self.root_dir = root_dir
        self.train=train
        if self.train == True:
            self.folder_path = self.root_dir + 'train/{0}/'.format(plane)
            self.records = pd.read_csv(
                self.root_dir + 'train-{0}.csv'.format(task), header=0, names=['id', 'label'])
        else:
            self.folder_path = self.root_dir + 'valid/{0}/'.format(plane)

            self.records = pd.read_csv(
                self.root_dir + 'valid-{0}.csv'.format(task), header=0, names=['id', 'label'])

        self.records['id'] = self.records['id'].map(
            lambda i: '0' * (4 - len(str(i))) + str(i))
        self.paths = [self.folder_path + filename +
                      '.npy' for filename in self.records['id'].tolist()]
        self.labels = self.records['label'].tolist()

        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        array = np.load(self.paths[index]) #load MRI
        label = self.labels[index] #get label of MRI
        label = torch.FloatTensor([label]) #convert type from numpy to torch

        # Apply transforms per slice, not to the whole volume
        if self.transform:
            slices = []
            for s in array:
                # Normalise to [0, 255] uint8 for PIL
                s_norm = ((s - s.min()) / (s.max() - s.min() + 1e-8) * 255).astype(np.uint8)
                pil_slice = Image.fromarray(s_norm).convert('RGB')  # [H, W, 3]
                slices.append(self.transform(pil_slice))             # [3, H, W] tensor
            array = torch.stack(slices, dim=0)                       # [N, 3, H, W]
        else:
            # Normalise and stack channels manually for validation
            array = (array - array.min()) / (array.max() - array.min() + 1e-8)
            array = np.stack((array,) * 3, axis=1)                   # [N, 3, H, W]
            array = torch.FloatTensor(array)

        return array, label


#Train the model
##Define variables
**TO DO:** Change directory to where you store your data. Use the toolbar to the side of this page to view your file system.

In [ ]:
directory='/content/gdrive/MyDrive/MRNET_Dataset/' #directory to the data
task = 'acl'
plane = 'sagittal'
lr = 1e-5 #learning rate (Problem 3/Fix 3: updated from 1e-4 to 1e-5)
num_epochs = 50 # number of epochs

# --- Experimentation Configuration ---
backbone = 'resnet18'      # Options: 'resnet18', 'resnet34', 'resnet50', 'efficientnet_b0'
pooling = 'attention'      # Options: 'attention', 'max', 'avg'
dropout_prob = 0.5         # Dropout probability to prevent overfitting
weight_decay = 1e-4        # Weight decay for L2 regularization
accumulation_steps = 8     # Gradient accumulation steps to simulate larger batch size
early_trigger = 10         # Epochs to wait for val AUC improvement before early stopping


##Initialise the model, optimiser, scheduler, transformations and data loader.

In [ ]:
model = Net(backbone=backbone, pooling=pooling, dropout_prob=dropout_prob) #initialise the model
if torch.cuda.is_available(): #if there is a GPU available, put the model on the GPU
    model = model.cuda()

# Stage 1: freeze backbone, only train attention + classifier (epochs 0-9)
for param in model.pretrained_model.parameters():
    param.requires_grad = False

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=weight_decay
)

# Solve Problem 3 / Fix 3: ReduceLROnPlateau scheduler based on val AUC in max mode (removed deprecated verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3)

#define a compose function that is a series of transformations on the images.
augmentor = transforms.Compose([
    transforms.RandomRotation(25),  # Randomly rotate the image
    transforms.RandomAffine(degrees=0, translate=(0.11, 0.11)),  # Randomly translated images
    transforms.RandomHorizontalFlip(),  # Randomly flip images
    transforms.ToTensor(),  # convert PIL image to tensor at the end!
])

#initialise the train and validation datasets
train_dataset = Dataset(directory, task, plane,
                         train=True, transform=augmentor)
valid_dataset = Dataset(
      directory, task, plane, train=False, transform=None)

# Solve Problem 2: Balanced loss weighting to avoid double counting class imbalance
pos = sum(train_dataset.labels)
neg = len(train_dataset.labels) - pos
pos_weight = torch.tensor([neg / pos])
if torch.cuda.is_available():
    pos_weight = pos_weight.cuda()

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Simpler train loader with shuffle=True and no sampler (resolves Bug 2)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=1, shuffle=True, num_workers=0, drop_last=False)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=1, shuffle=False, num_workers=0, drop_last=False)


##Training Loop

In [ ]:
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

early_stop = 0 #counter for the number of iterations where there has been no increase in validation AUC
best_val_auc = 0

#for loop for each epoch
for epoch in range(num_epochs):
      # Stage transition at epoch 10 (resolves Bug 3: Two-stage training)
      if epoch == 10:
          print("Transitioning to Stage 2: Unfreezing backbone and setting differential learning rates.")
          for param in model.pretrained_model.parameters():
              param.requires_grad = True
          
          # Re-initialize optimizer with differential learning rates
          params_to_optimize = [
              {'params': model.pretrained_model.parameters(), 'lr': 1e-5},
              {'params': model.classifier.parameters(),       'lr': 1e-4},
          ]
          if hasattr(model, 'attention'):
              params_to_optimize.append(
                  {'params': model.attention.parameters(),    'lr': 1e-4}
              )
          optimizer = optim.Adam(params_to_optimize, weight_decay=weight_decay)
          
          # Re-initialize scheduler to use the new optimizer
          scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
              optimizer, mode='max', factor=0.5, patience=3
          )

      y_preds = []
      y_trues = []
      losses = []

      # Solve Fix 1: Switch to train mode at start of training epoch
      model.train()

      optimizer.zero_grad()
      #loop through each MRI in the training set
      for i, (image, label) in enumerate(train_loader):

          #load all data onto the GPU
          if torch.cuda.is_available():
              image = image.cuda()
              label = label.cuda()

          #pass the MRI through the model and ensure 1D shape
          prediction = model.forward(image.float()).view(-1)
          label = label.view(-1)

          #calculate the loss using criterion (resolves Bug 2: Double-Counting)
          loss = criterion(prediction, label)

          # normalize loss to account for batch accumulation
          loss = loss / accumulation_steps
          loss.backward() #back propagation

          # step optimizer only every accumulation_steps or at the end of the epoch
          if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
              optimizer.step()
              optimizer.zero_grad()

          loss_value = loss.item() * accumulation_steps
          losses.append(loss_value)

          probas = torch.sigmoid(prediction)

          y_trues.append(int(label.item()))
          y_preds.append(probas.item())

      train_loss = np.round(np.mean(losses), 4)
      try:
          train_auc = np.round(metrics.roc_auc_score(y_trues, y_preds), 4)
      except Exception as e:
          train_auc = 0.5

      #evaluate the model on the validation data after each epoch
      # Solve Fix 1: Switch to eval mode before validation
      model.eval()
      y_trues = []
      y_preds = []
      val_loss = 0.0

      with torch.no_grad():
          for i, (image, label) in enumerate(valid_loader):

            if torch.cuda.is_available():
                image = image.cuda()
                label = label.cuda()

            #pass the MRI through the model and ensure 1D shape
            prediction = model.forward(image.float()).view(-1)
            label = label.view(-1)

            loss = criterion(prediction, label)
            val_loss += loss.item()

            probas = torch.sigmoid(prediction)

            y_trues.append(int(label.item()))
            y_preds.append(probas.item())

      # Solve Fix 2: validation loss averaged over batches (not sum)
      val_loss /= len(valid_loader)
      val_loss = np.round(val_loss, 4)
      try:
          val_auc = np.round(metrics.roc_auc_score(y_trues, y_preds), 4)
      except Exception as e:
          val_auc = 0.5

      if val_auc > best_val_auc:
        best_val_auc = val_auc
        early_stop=0
      else:
        early_stop+= 1

      # Solve Fix 3: Step scheduler using val_auc instead of val_loss
      scheduler.step(val_auc)

      print("epoch : {0} | train loss : {1} | train auc {2} | val loss {3} | val auc {4} ".format(
          epoch, train_loss, train_auc, val_loss, val_auc))
      print('-' * 30)

      if early_stop == early_trigger:
        print('Early stopping after {} epochs'.format(epoch))
        break


epoch : 0 | train loss : 1.0567 | train auc 0.791 | val loss 46.8776 | val auc 0.5 
------------------------------
epoch : 1 | train loss : 1.0656 | train auc 0.7928 | val loss 30.7128 | val auc 0.5171 
------------------------------
epoch : 2 | train loss : 1.0706 | train auc 0.7868 | val loss 33.991 | val auc 0.4654 
------------------------------
epoch : 3 | train loss : 1.0556 | train auc 0.7998 | val loss 42.0696 | val auc 0.5 
------------------------------
epoch : 4 | train loss : 0.9782 | train auc 0.8272 | val loss 32.5939 | val auc 0.4521 
------------------------------
epoch : 5 | train loss : 1.0656 | train auc 0.7998 | val loss 29.1644 | val auc 0.5293 
------------------------------
epoch : 6 | train loss : 1.0 | train auc 0.8174 | val loss 40.5471 | val auc 0.4858 
------------------------------
epoch : 7 | train loss : 1.0086 | train auc 0.8213 | val loss 38.89 | val auc 0.4983 
------------------------------
epoch : 8 | train loss : 0.9805 | train auc 0.829 | val loss 